# Bulk editor for ArcGIS Online Item Details pages


**Welcome!**  

This notebook helps you scan, review, and update ArcGIS Online items at scale. It focuses on the Terms of Use section, stored in the `licenseInfo` field, and looks for text or HTML that you may want to replace.

This version bundles `helper_functions.py` and `Esri_ToU.html` template directly into the notebook, so when running Step 1 those files will be expanded into a new folder and saved into `/arcgis/home/notebook_outputs`. You will be able to modify both input and output files as you progress. A review webpage is produced that lets you see what will change before you make any edits, and you can selectively choose to edit items from the report.

*** BE CAUTIOUS WITH ANY TOOL LIKE THIS THAT BULK EDITS ITEMS *** However, you will have plenty of chances to review the work before commiting any changes.

**Where this notebook can run**  
- ArcGIS Online Notebook (JupyterLab-style).
- VS Code on macOS with a local Jupyter kernel.
- VS Code on Windows with a local Jupyter kernel.

**How to use this notebook**  
 - Click on the text "Setup and authenticate" below. 
 - There are two types of cells, Markdown (formatted notes) and Code.
 - An indicator -- typically a vertical blue line -- should highlight that you have selected the "Setup and authenticate" Markdown cell.
 - Once selected, click the "Play" button in the toolbar above to run the cell and advance to the next Code cell.
 - Click the "Play" button a second time to run the code cell.
 - After several seconds a "Setup Notebook" button should appear. Click the button to begin setup and authentication.
 - After each cell completes, click the text within the following Markdown cell.
 - Click the "Play" button to advance to the Code cell, then click the "Play" button a second time to make a button appear.
 - Click the button to run the code in the cell. 

**Notes**  
- Organization-wide scans can take time, especially in large orgs, so progress messages are shown as users are processed.
- You can monitor the status of long running cells by viewing the small circle in the top right of the page.
- If you click on a code cell it will expand showing you the behind-the-scenes Python code.
- For a cleaner interface select View > Collapse All Code in the menu bar above to hide the code .
- If at any point you get stuck and want to start over, just click Kernel > Restart Kernel and Clear Outputs of All Cells... in the menu bar
- The workflow is designed to be safe by default: review first, then update.


**TL;DR**


In [ ]:
# Run this cell to display Notebook details
from IPython.display import display, Markdown

# Display details of what this notebook does
tldr_md = """
**What this notebook does**  
- Authenticates to ArcGIS Online
- Scans an entire Organization's Item Details page's "Terms of Use" section (aka `licenseInfo`).
- Finds items that match one or more target strings (for example, outdated logo URLs or legacy text snippets).
- Exports scan outputs for auditability (default filenames: `scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans to target additional terms while excluding already matched item IDs. (improves scan speed and accuracy)
- Builds a dry-run update plan that shows old vs new HTML before any edit is applied.
- Generates a user-friendly side-by-side HTML review report for visual QA.
- Applies updates only after explicit confirmation, then exports success and error results.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.
"""
display(Markdown(tldr_md))

## 1. Setup and authenticate
Write the bundled helper files into the runtime, then initialize the notebook environment and connect to ArcGIS Online.


In [5]:
# Bootstrap bundled assets for the portable notebook.import base64import sysfrom pathlib import PathOUTPUT_DIR_NAME = "notebook_outputs"RUNTIME_ROOT = Path("/arcgis/home") if Path("/arcgis/home").exists() else Path.cwd()RUNTIME_DIR = (RUNTIME_ROOT / OUTPUT_DIR_NAME).resolve()RUNTIME_DIR.mkdir(parents=True, exist_ok=True)HELPER_FUNCTIONS_B64 = (    "..."    "X2Rm")ESRI_TOU_HTML_B64 = (    "PHA+CiAgICA8aW1nIHNyYz0iaHR0cHM6Ly93d3cuZXNyaS5jb20vY29udGVudC9kYW0vZXNyaXNpdGVzL2VuLXVzL2NvbW1vbi9sb2dvcy9lc3JpLWxvZ28u"    "anBnIiBhbHQ9IkVzcmkgbG9nbyIgd2lkdGg9IjExMyIgaGVpZ2h0PSIzOSI+CjwvcD4KPHAgc3R5bGU9ImNvbG9yOnJnYig3NCw3NCw3NCk7Zm9udC1mYW1p"    "bHk6J0F2ZW5pciBOZXh0JyxBdmVuaXIsJ0hlbHZldGljYSBOZXVlJyxzYW5zLXNlcmlmO2ZvbnQtc2l6ZToxNnB4O21hcmdpbjowIDAgMXJlbTsiPgogICAg"    "VGhpcyB3b3JrIGlzIGxpY2Vuc2VkIHVuZGVyIHRoZSBFc3JpIE1hc3RlciBMaWNlbnNlIEFncmVlbWVudC4KPC9wPgo8cCBzdHlsZT0iY29sb3I6cmdiKDc0"    "LDc0LDc0KTtmb250LWZhbWlseTonQXZlbmlyIE5leHQnLEF2ZW5pciwnSGVsdmV0aWNhIE5ldWUnLHNhbnMtc2VyaWY7Zm9udC1zaXplOjE2cHg7bWFyZ2lu"    "OjA7Ij4KICAgIDxhIHN0eWxlPSJjb2xvcjpyZ2IoMCw5NywxNTUpO3RleHQtZGVjb3JhdGlvbjpub25lOyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVu"    "ZXIgbm9yZWZlcnJlciIgaHJlZj0iaHR0cHM6Ly9nb3RvLmFyY2dpcy5jb20vdGVybXNvZnVzZS92aWV3c3VtbWFyeSI+PHN0cm9uZz5WaWV3IFN1bW1hcnk8"    "L3N0cm9uZz48L2E+IHwgPGEgc3R5bGU9ImNvbG9yOnJnYigwLDk3LDE1NSk7dGV4dC1kZWNvcmF0aW9uOm5vbmU7IiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJu"    "b29wZW5lciBub3JlZmVycmVyIiBocmVmPSJodHRwczovL2dvdG8uYXJjZ2lzLmNvbS90ZXJtc29mdXNlL3ZpZXd0ZXJtc29mdXNlIj48c3Ryb25nPlZpZXcg"    "VGVybXMgb2YgVXNlPC9zdHJvbmc+PC9hPgo8L3A+")BOOTSTRAP_FILES = {    "helper_functions.py": base64.b64decode(HELPER_FUNCTIONS_B64).decode("utf-8"),    "Esri_ToU.html": base64.b64decode(ESRI_TOU_HTML_B64).decode("utf-8"),}for filename, file_text in BOOTSTRAP_FILES.items():    target_path = RUNTIME_DIR / filename    target_path.write_text(file_text, encoding="utf-8")    print(f"Bootstrapped asset: {target_path}")if str(RUNTIME_DIR) not in sys.path:    sys.path.insert(0, str(RUNTIME_DIR))print(f"Portable notebook assets are ready in: {RUNTIME_DIR}")# When running in ArcGIS Online, you can select View > Collapse All Code in the toolbar above to hide the code for a cleaner view.
print("Initializing...")

# Cell 1. Import packages, configure paths, authenticate, and load helper functions
import sys
from pathlib import Path
import pandas as pd
import ipywidgets as widgets

NOTEBOOK_DIR = Path.cwd().resolve()
IS_PORTABLE_NOTEBOOK = "RUNTIME_DIR" in globals()

if IS_PORTABLE_NOTEBOOK:
    # Portable notebook: prefer freshly bootstrapped assets in notebook_outputs.
    candidate_helper_dirs = [
        Path(globals()["RUNTIME_DIR"]).resolve(),
        NOTEBOOK_DIR / "notebook_outputs",
        NOTEBOOK_DIR,
        Path("/arcgis/home/notebook_outputs"),
        Path("/arcgis/home"),
    ]
else:
    # Source notebook: prefer repository files first to avoid stale bootstrapped copies.
    candidate_helper_dirs = [
        NOTEBOOK_DIR,
        NOTEBOOK_DIR / ".local_testing" / "notebook_outputs",
        Path("/arcgis/home/notebook_outputs"),
        Path("/arcgis/home"),
    ]

selected_helper_dir = None
for p in candidate_helper_dirs:
    helper_file = p / "helper_functions.py"
    if helper_file.exists():
        selected_helper_dir = p.resolve()
        break

if selected_helper_dir is None:
    raise FileNotFoundError(
        "Could not locate helper_functions.py in expected locations. "
        "For source notebook runs, keep helper_functions.py in the notebook folder. "
        "For portable runs, execute the bootstrap section first."
    )

# Ensure the selected helper path wins over stale entries.
selected_helper_dir_str = str(selected_helper_dir)
sys.path[:] = [x for x in sys.path if x != selected_helper_dir_str]
sys.path.insert(0, selected_helper_dir_str)

# Force reload source if helper_functions was previously imported from another location.
if "helper_functions" in sys.modules:
    del sys.modules["helper_functions"]

from helper_functions import (
    detect_environment,
    default_output_dir_str,
    default_output_path_str,
    initialize_ui,
    set_runtime_context,
    bind_button_with_status,
    bind_primary_scan_with_cancel,
    setup_notebook_btn,
    save_scan_outputs_btn,
    save_secondary_scan_outputs_btn,
    load_previous_scan_btn,
    run_secondary_scan_btn,
    run_strict_match_filter_btn,
    run_dry_run_with_file_btn,
    create_report_btn,
    export_dry_run_btn,
    load_update_selection_btn,
    apply_updates_btn,
    export_final_results_btn,
    OFFICIAL_TOU_HTML_FILE,
)

# Resolve the canonical ToU template path based on notebook mode.
if IS_PORTABLE_NOTEBOOK:
    resolved_tou_path = (selected_helper_dir / "Esri_ToU.html").resolve()
else:
    resolved_tou_path = (NOTEBOOK_DIR / "Esri_ToU.html").resolve()
if not resolved_tou_path.exists():
    resolved_tou_path = Path(OFFICIAL_TOU_HTML_FILE).resolve()

# Set Pandas dataframe display options
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 1000)

# Define shared notebook state
context = {
    "gis": None,
    "matches_df": None,
    "errors_df": None,
    "all_items_df": None,
    "TARGET_STRINGS": [],
    "output_dir": default_output_dir_str(),
    "official_tou_html_file": str(resolved_tou_path),
}
set_runtime_context(context)

# Detect the current environment
current_env, env_string = detect_environment()
print(f"Currently running in {env_string}")
print(f"Default output folder: {context['output_dir']}")

if not IS_PORTABLE_NOTEBOOK:
    print(f"Helper path: {selected_helper_dir}")
    print(f"Official ToU template file: {context['official_tou_html_file']}")

output1 = initialize_ui("output")
context["output1"] = output1
auth_container = widgets.VBox([])
context["auth_container"] = auth_container

# Create widgets
btn1 = initialize_ui("button", description="Setup Notebook", width="200px")
status1 = widgets.HTML(value="")
context["status1"] = status1
display(widgets.HBox([btn1, status1]))
bind_button_with_status(
    btn1,
    setup_notebook_btn,
    "status1",
    "Setup in progress... please wait.",
    "Setup complete.",
    "Setup failed. See details below.",
    output_key="output1",
)
display(output1)
display(auth_container)

Initializing...
Currently running in VSCode Notebook environment
Default output folder: /Users/davi6569/Documents/GitHub/AGO-item-description-editor/.local_testing/notebook_outputs
Helper path: /Users/davi6569/Documents/GitHub/AGO-item-description-editor
Official ToU template file: /Users/davi6569/Documents/GitHub/AGO-item-description-editor/Esri_ToU.html


Output()

VBox()

## 2. Scan your content
Search your content for the text strings and/or HTML entered below.
There is an optional cap: leave it blank to scan the entire org, or enter a number to stop after that many matches for faster test runs.
After the scan finishes, optional save fields appear below when there is scan output worth exporting.


In [ ]:
# Cell 2: Scan org content for licenseInfo matches and optionally save the results
output2 = initialize_ui("output")
context["output2"] = output2

help2 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "<strong>Enter text or HTML to find in the Terms of Use section.</strong> "
        "Use CSV-style input (comma-separated).<br>"
        "Wrap text with spaces in double quotes, for example: "
        "&quot;&lt;a href=https://example.com&gt; link &lt;/a&gt;&quot;.<br>"
        "Formatting will automatically be bundled for processing when you click the button."
        "</div>"
    )
)

input2 = initialize_ui(
    "textarea",
    value='https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png, esrilogo',
    description="",
    width="700px",
    height="70px",
)
context["input2"] = input2
input2_limit = initialize_ui(
    "text",
    value="",
    description="Stop scan after... (optional):",
    width="300px",
)
context["input2_limit"] = input2_limit
btn2 = initialize_ui("button", description="Scan for items", width="200px")
status2 = widgets.HTML(value="")
context["status2"] = status2

output3 = initialize_ui("output")
context["output3"] = output3
input3_matches = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Save matching items to CSV:",
    width="700px",
)
context["input3_matches"] = input3_matches
input3_errors = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Save errors to CSV:",
    width="700px",
)
context["input3_errors"] = input3_errors
input3_all_items = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="Save scanned items to CSV:",
    width="700px",
)
context["input3_all_items"] = input3_all_items
btn3 = initialize_ui("button", description="Save files")
status3 = widgets.HTML(value="")
context["status3"] = status3
save3_container = widgets.VBox([])
context["save3_container"] = save3_container

def refresh_scan_save_ui():
    matches_df = context.get("matches_df")
    errors_df = context.get("errors_df")
    all_items_df = context.get("all_items_df")

    if matches_df is None and errors_df is None and all_items_df is None:
        save3_container.children = ()
        return

    step3_children = [widgets.HTML(value="<div style='margin-top:12px; font-weight:600;'>Optional: Save current scan outputs</div>")]
    if matches_df is not None and not matches_df.empty:
        step3_children.append(input3_matches)
    if errors_df is not None and not errors_df.empty:
        step3_children.append(input3_errors)
    if all_items_df is not None and not all_items_df.empty:
        step3_children.append(input3_all_items)

    if len(step3_children) == 1:
        step3_children.append(
            widgets.HTML(
                value="<div style='margin:0; padding:0;'>No non-empty scan output files are available to save.</div>"
            )
        )
    else:
        step3_children.append(widgets.HBox([btn3, status3]))

    step3_children.append(output3)
    save3_container.children = tuple(step3_children)

context["refresh_scan_save_ui"] = refresh_scan_save_ui
refresh_scan_save_ui()

display(
    widgets.VBox([
        help2,
        input2,
        input2_limit,
        widgets.HBox([btn2, status2]),
        output2,
        save3_container,
    ])
)

bind_primary_scan_with_cancel(
    btn2,
    status_key="status2",
    output_key="output2",
    input_key="input2",
    limit_input_key="input2_limit",
)

bind_button_with_status(
    btn3,
    save_scan_outputs_btn,
    "status3",
    "Save in progress... please wait.",
    "Save complete.",
    "Save failed. See details below.",
    output_key="output3",
)

## 3. Optionally reload results from a previous scan
Load previously saved CSV files so you can continue working without running a new scan.


In [ ]:
# Cell 3: Optionally load results from a previous scan
output4 = initialize_ui("output")
context["output4"] = output4

input4_matches = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Matches CSV:",
    width="900px",
)
context["input4_matches"] = input4_matches
input4_errors = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Errors CSV:",
    width="900px",
)
context["input4_errors"] = input4_errors
input4_all_items = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="All items CSV:",
    width="900px",
)
context["input4_all_items"] = input4_all_items
btn4 = initialize_ui("button", description="Load previous scan files", width="240px")
status4 = widgets.HTML(value="")
context["status4"] = status4

display(
    widgets.VBox([
        input4_matches,
        input4_errors,
        input4_all_items,
        widgets.HBox([btn4, status4]),
        output4,
    ])
)

bind_button_with_status(
    btn4,
    load_previous_scan_btn,
    "status4",
    "Load in progress... please wait.",
    "Load complete.",
    "Load failed. See details below.",
    output_key="output4",
)

## 4. Secondary scan
This step saves you time on the primary run if you want to search additional terms.
If you want to search again, click the check box and add the new terms to the input box.
After the secondary scan finishes, optional save fields appear below when there is secondary scan output worth exporting.


In [ ]:
# Cell 4: Optional secondary scan using new terms and optionally save the results
output5 = initialize_ui("output")
context["output5"] = output5
checkbox5 = initialize_ui("checkbox", description="Run secondary scan with new search terms?", value=False)
context["checkbox5"] = checkbox5
help5 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "As above, use CSV-style input separated by commas.<br>"
        "Wrap text with spaces in double quotes, for example: &quot;text with spaces&quot;."
        "</div>"
    )
)
input5 = initialize_ui(
    "textarea",
    value='https://www.esri.com/content/dam/esrisites/en-us/common/logos/esri-logo.jpg',
    description="",
    width="700px",
    height="70px",
)
context["input5"] = input5

btn5 = initialize_ui("button", description="Run secondary scan")
status5 = widgets.HTML(value="")
context["status5"] = status5

output6 = initialize_ui("output")
context["output6"] = output6
input6_secondary_matches = initialize_ui(
    "text",
    value=default_output_path_str("secondary_scan_matches.csv"),
    description="Secondary matches CSV:",
    width="700px",
)
context["input6_secondary_matches"] = input6_secondary_matches
input6_secondary_errors = initialize_ui(
    "text",
    value=default_output_path_str("secondary_scan_errors.csv"),
    description="Secondary errors CSV:",
    width="700px",
)
context["input6_secondary_errors"] = input6_secondary_errors
input6_secondary_all_items = initialize_ui(
    "text",
    value=default_output_path_str("secondary_scan_all_items.csv"),
    description="Secondary all items CSV:",
    width="700px",
)
context["input6_secondary_all_items"] = input6_secondary_all_items
btn6 = initialize_ui("button", description="Save secondary files")
status6 = widgets.HTML(value="")
context["status6"] = status6
save6_container = widgets.VBox([])
context["save6_container"] = save6_container

def refresh_secondary_save_ui():
    new_matches_df = context.get("new_matches_df")
    new_errors_df = context.get("new_errors_df")
    new_all_items_df = context.get("new_all_items_df")

    if new_matches_df is None and new_errors_df is None and new_all_items_df is None:
        save6_container.children = ()
        return

    step6_children = [widgets.HTML(value="<div style='margin-top:12px; font-weight:600;'>Optional: Save current secondary scan outputs</div>")]
    if new_matches_df is not None and not new_matches_df.empty:
        step6_children.append(input6_secondary_matches)
    if new_errors_df is not None and not new_errors_df.empty:
        step6_children.append(input6_secondary_errors)
    if new_all_items_df is not None and not new_all_items_df.empty:
        step6_children.append(input6_secondary_all_items)

    if len(step6_children) == 1:
        step6_children.append(
            widgets.HTML(
                value="<div style='margin:0; padding:0;'>No non-empty secondary scan output files are available to save.</div>"
            )
        )
    else:
        step6_children.append(widgets.HBox([btn6, status6]))

    step6_children.append(output6)
    save6_container.children = tuple(step6_children)

context["refresh_secondary_save_ui"] = refresh_secondary_save_ui
refresh_secondary_save_ui()

display(
    widgets.VBox([
        checkbox5,
        help5,
        input5,
        widgets.HBox([btn5, status5]),
        output5,
        save6_container,
    ])
)

bind_button_with_status(
    btn5,
    run_secondary_scan_btn,
    "status5",
    "Secondary scan in progress... please wait.",
    "Secondary scan complete.",
    "Secondary scan failed. See details below.",
    output_key="output5",
)

bind_button_with_status(
    btn6,
    save_secondary_scan_outputs_btn,
    "status6",
    "Secondary save in progress... please wait.",
    "Secondary save complete.",
    "Secondary save failed. See details below.",
    output_key="output6",
)

## 5. Optionally narrow your original query
You can filter the scan results to show only items containing the exact text entered below.


In [ ]:
# Cell 5: Optionally filter the scan result to rows containing the exact text below
output7 = initialize_ui("output")
context["output7"] = output7
input7 = initialize_ui(
    "text",
    value="https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png",
    description="Exact text:",
    width="700px",
)
context["input7"] = input7
btn7 = initialize_ui("button", description="Filter exact matches", width="200px")
status7 = widgets.HTML(value="")
context["status7"] = status7
display(widgets.VBox([input7, widgets.HBox([btn7, status7]), output7]))

bind_button_with_status(
    btn7,
    run_strict_match_filter_btn,
    "status7",
    "Filter in progress... please wait.",
    "Filter complete.",
    "Filter failed. See details below.",
    output_key="output7",
)

## 6. Dry run
Do a dry-run before making any changes. Modify the input file to use your own custom HTML that will replace the search terms.
By default, the edit logic uses a semi-greedy matcher: it looks for a recognized Esri logo, then scans forward within bounded distance for the license text and the related links. This makes the workflow tolerant of minor formatting differences, but it can match a broader block of HTML than a literal exact match.
If you enable strict matching below, the dry run will only replace blocks that contain the recognized logo, license text, summary link, and terms link in the expected order with tighter bounds between them. Use strict mode when you want a more conservative replacement plan.
After the dry run finishes, an optional CSV export field appears below when there is a dry-run plan worth saving.


In [ ]:
# Cell 6: Do a dry-run before making any changes and optionally export the plan
output8 = initialize_ui("output")
context["output8"] = output8
input8 = initialize_ui(
    "text",
    value=context.get("official_tou_html_file", OFFICIAL_TOU_HTML_FILE),
    description="Input HTML file:",
    width="900px",
)
context["input8"] = input8
checkbox8 = initialize_ui(
    "checkbox",
    description="Enforce strict matching?",
    value=False,
    width="320px",
)
context["checkbox8"] = checkbox8
btn8 = initialize_ui("button", description="Do a dry run", width="180px")
status8 = widgets.HTML(value="")
context["status8"] = status8

output9 = initialize_ui("output")
context["output9"] = output9
input9_csv_name = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_results.csv"),
    description="Output CSV:",
    width="700px",
)
context["input9_csv_name"] = input9_csv_name
btn9 = initialize_ui("button", description="Export dry-run CSV", width="200px")
status9 = widgets.HTML(value="")
context["status9"] = status9
export9_container = widgets.VBox([])
context["export9_container"] = export9_container

def refresh_dry_run_export_ui():
    plan_df = context.get("plan_df")

    if plan_df is None:
        export9_container.children = ()
        return

    export9_container.children = (
        widgets.HTML(value="<div style='margin-top:12px; font-weight:600;'>Optional: Export current dry-run results</div>"),
        input9_csv_name,
        widgets.HBox([btn9, status9]),
        output9,
    )

context["refresh_dry_run_export_ui"] = refresh_dry_run_export_ui
refresh_dry_run_export_ui()

display(widgets.VBox([input8, checkbox8, widgets.HBox([btn8, status8]), output8, export9_container]))

bind_button_with_status(
    btn8,
    run_dry_run_with_file_btn,
    "status8",
    "Dry run in progress... please wait.",
    "Dry run complete.",
    "Dry run failed. See details below.",
    output_key="output8",
)

bind_button_with_status(
    btn9,
    export_dry_run_btn,
    "status9",
    "Dry-run export in progress... please wait.",
    "Dry-run export complete.",
    "Dry-run export failed. See details below.",
    output_key="output9",
)

## 7. Create a report to review before committing the edits
A HTML file will be created. Large reports cannot be properly displayed in the output cell. When this happens, download the file from /arcgis/home/notebook_outputs by clicking on the filename. Then open that file in your local browser. You'll then make selections, save a JSON file and upload that file to /arcgis/home/notebook_outputs for the final editing step.
There is an optional cap: leave it blank to include all planned updates, or enter a number to generate a smaller review report for faster testing.


In [ ]:
# Cell 7: Generate an HTML report for review before updating items
output10 = initialize_ui("output")
context["output10"] = output10
input10_report_name = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_report.html"),
    description="Report HTML:",
    width="700px",
)
context["input10_report_name"] = input10_report_name
input10_selection_json = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="Filename generated when downloading IDs after review:",
    width="700px",
)
context["input10_selection_json"] = input10_selection_json
btn10 = initialize_ui("button", description="Create report", width="200px")
status10 = widgets.HTML(value="")
context["status10"] = status10

display(
    widgets.VBox([
        input10_report_name,
        input10_selection_json,
        widgets.HBox([btn10, status10]),
        output10,
    ])
)

bind_button_with_status(
    btn10,
    create_report_btn,
    "status10",
    "Report generation in progress... please wait.",
    "Report generation complete.",
    "Report generation failed. See details below.",
    output_key="output10",
)

## 8. Commit updates
Use this step to safely confirm what will be edited before making any changes.
1. Enter the JSON or CSV file path that contains item IDs selected from the report. (the default file will be preloaded)
2. Click **Load file** to preview how many items will be edited.
3. Review the summary shown in the output area.
4. If it looks correct, type `APPLY UPDATES` in the confirmation box.
5. Click **Execute update** to apply the edits.


In [ ]:
# Cell 8: Apply updates only after reviewing the dry run report 
output11 = initialize_ui("output")
context["output11"] = output11
input11_ids = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="JSON file with item IDs to update:",
    width="700px",
)
context["input11_ids"] = input11_ids

btn11_load = initialize_ui("button", description="Load file", width="140px")
status11_load = widgets.HTML(value="")
context["status11_load"] = status11_load

input11_confirm = initialize_ui(
    "text",
    value="",
    description="Type APPLY UPDATES to confirm:",
    width="700px",
)
context["input11_confirm"] = input11_confirm

btn11 = initialize_ui("button", description="Execute update", width="180px")
status11 = widgets.HTML(value="")
context["status11"] = status11
display(
    widgets.VBox([
        input11_ids,
        widgets.HBox([btn11_load, status11_load]),
        output11,
        input11_confirm,
        widgets.HBox([btn11, status11]),
    ])
)

bind_button_with_status(
    btn11_load,
    load_update_selection_btn,
    "status11_load",
    "Loading file and previewing selection... please wait.",
    "File loaded. Review summary below.",
    "Load failed. See details below.",
    output_key="output11",
)

bind_button_with_status(
    btn11,
    apply_updates_btn,
    "status11",
    "Update in progress... please wait.",
    "Update complete.",
    "Update failed. See details below.",
    output_key="output11",
)

## 9. Export results of the editing process
Export csv files for record-keeping.


In [ ]:
# Cell 9: Export final update results to CSV files for record-keeping
output12 = initialize_ui("output")
context["output12"] = output12
input12_success_csv = initialize_ui(
    "text",
    value=default_output_path_str("update_successes.csv"),
    description="Success CSV:",
    width="700px",
)
context["input12_success_csv"] = input12_success_csv
input12_errors_csv = initialize_ui(
    "text",
    value=default_output_path_str("update_errors.csv"),
    description="Errors CSV:",
    width="700px",
)
context["input12_errors_csv"] = input12_errors_csv
btn12 = initialize_ui("button", description="Export final CSVs", width="180px")
status12 = widgets.HTML(value="")
context["status12"] = status12

success_df = context.get("success_df")
update_errors_df = context.get("update_errors_df")

step12_children = []
if success_df is not None and not success_df.empty:
    step12_children.append(input12_success_csv)
if update_errors_df is not None and not update_errors_df.empty:
    step12_children.append(input12_errors_csv)

if not step12_children:
    step12_children.append(
        widgets.HTML(
            value="<div style='margin:0; padding:0;'>No non-empty final result files are available to export yet.</div>"
        )
    )
else:
    step12_children.append(widgets.HBox([btn12, status12]))

step12_children.append(output12)
display(widgets.VBox(step12_children))

bind_button_with_status(
    btn12,
    export_final_results_btn,
    "status12",
    "Final export in progress... please wait.",
    "Final export complete.",
    "Final export failed. See details below.",
    output_key="output12",
)